# Calculate effective coverage by quintile and scenario

Also by age, sex, and pregnancy status (though coverage will not vary by pregnancy status due to a lack of data).

This is similar to what the pregnancy simulation does at the individual level, but using groups instead.
It can be shared between multiplication models that do not incorporate individual heterogeneity.

In [1]:
import pandas as pd

In [2]:
location = "nigeria"
vehicle = "rice"
fortificant = "iron"

In [3]:
# Parameters
location = "india"
fortificant = "folate"
vehicle = "rice"


In [4]:
results_dir = f"../results/{fortificant}/{vehicle}"

In [5]:
full_coverage_probability = pd.read_csv(
    f"{results_dir}/baseline_fortification/full_coverage/{location}.csv"
)
full_coverage_probability = full_coverage_probability.set_index(
    [c for c in full_coverage_probability.columns if c != "value"]
).value
full_coverage_probability

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.108446
                            2                rice            0.139914
                            3                rice            0.125927
                            4                rice            0.112437
                            5                rice            0.081985
        5          15       1                rice            0.113265
                            2                rice            0.139304
                            3                rice            0.124909
                            4                rice            0.112687
                            5                rice            0.079338
        15         30       1                rice            0.114477
                            2                rice            0.150291
                            3                rice            0.129093
                            4   

In [6]:
any_coverage_probability = pd.read_csv(
    f"{results_dir}/baseline_fortification/any_coverage/{location}.csv"
)
any_coverage_probability = any_coverage_probability.set_index(
    [c for c in any_coverage_probability.columns if c != "value"]
).value
any_coverage_probability

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.624237
                            2                rice            0.584066
                            3                rice            0.554623
                            4                rice            0.559809
                            5                rice            0.346473
        5          15       1                rice            0.733094
                            2                rice            0.688794
                            3                rice            0.646130
                            4                rice            0.617076
                            5                rice            0.368363
        15         30       1                rice            0.664081
                            2                rice            0.608319
                            3                rice            0.577004
                            4   

In [7]:
partial_coverage_mean = pd.read_csv(
    f"{results_dir}/baseline_fortification/partial_coverage_amount/mean/{location}.csv"
)
partial_coverage_mean = partial_coverage_mean.set_index(
    [c for c in partial_coverage_mean.columns if c != "value"]
).value
partial_coverage_mean

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.529705
                            2                rice            0.539072
                            3                rice            0.534043
                            4                rice            0.504590
                            5                rice            0.472097
        5          15       1                rice            0.561247
                            2                rice            0.559741
                            3                rice            0.553740
                            4                rice            0.530998
                            5                rice            0.483357
        15         30       1                rice            0.532870
                            2                rice            0.549423
                            3                rice            0.550596
                            4   

In [8]:
current_coverage = (
    full_coverage_probability
    + (any_coverage_probability - full_coverage_probability) * partial_coverage_mean
)
current_coverage

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.381663
                            2                rice            0.379344
                            3                rice            0.354869
                            4                rice            0.338176
                            5                rice            0.206849
        5          15       1                rice            0.461142
                            2                rice            0.446876
                            3                rice            0.413530
                            4                rice            0.380517
                            5                rice            0.219041
        15         30       1                rice            0.407345
                            2                rice            0.401942
                            3                rice            0.375711
                            4   

In [9]:
from lsff_utils import config_utils

scenarios = config_utils.get_intervention_scenarios(location, vehicle)
scenarios

['intervention']

In [10]:
any_consumption = pd.read_csv(
    f"../results/{vehicle}/vehicle_consumption/any/{location}.csv"
)
any_consumption = any_consumption.set_index(
    [c for c in any_consumption.columns if c != "value"]
).value
any_consumption

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.890088
                            2                rice            0.875116
                            3                rice            0.873100
                            4                rice            0.884775
                            5                rice            0.865564
        5          15       1                rice            0.961146
                            2                rice            0.980478
                            3                rice            0.989830
                            4                rice            0.992594
                            5                rice            0.988565
        15         30       1                rice            0.976640
                            2                rice            0.970350
                            3                rice            0.988181
                            4   

In [11]:
if (
    len(
        any_consumption.reset_index()[["sex", "age_start", "age_end"]].drop_duplicates()
    )
    == 1
):
    # Assumed same for all ages/sexes
    any_consumption = any_consumption.droplevel(["sex", "age_start", "age_end"])
    display(any_consumption)

In [12]:
fortifiability = pd.read_csv(
    f"../results/{vehicle}/vehicle_consumption/fortifiability/{location}.csv"
)
fortifiability = fortifiability.set_index(
    [c for c in fortifiability.columns if c != "value"]
).value
fortifiability

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.647826
                            2                rice            0.656178
                            3                rice            0.656911
                            4                rice            0.656052
                            5                rice            0.594069
        5          15       1                rice            0.708132
                            2                rice            0.711168
                            3                rice            0.708884
                            4                rice            0.689265
                            5                rice            0.609102
        15         30       1                rice            0.647649
                            2                rice            0.664128
                            3                rice            0.676266
                            4   

In [13]:
import pathlib, numpy as np

for scenario in scenarios:
    intervention_coverage = pd.read_csv(
        f"{results_dir}/{scenario}/intervention_fortification/any_coverage/{location}.csv"
    )
    intervention_coverage = intervention_coverage.set_index(
        [c for c in intervention_coverage.columns if c != "value"]
    ).value
    target_coverage = intervention_coverage * fortifiability
    display(target_coverage)
    assert (
        (target_coverage > current_coverage.reindex_like(target_coverage))
        | np.isclose(target_coverage, current_coverage.reindex_like(target_coverage))
    ).all()
    target_coverage[
        np.isclose(target_coverage, current_coverage.reindex_like(target_coverage))
    ] = current_coverage.reindex_like(target_coverage)
    # Not all coverage is effective -- this is as a proportion of coverage!
    effectiveness = pd.read_csv(
        f"{results_dir}/{scenario}/intervention_fortification/effectiveness/{location}.csv"
    )
    effectiveness = effectiveness.set_index(
        [c for c in effectiveness.columns if c != "value"]
    ).value
    effective_intervention_coverage = any_consumption * target_coverage * effectiveness
    path = f"{results_dir}/{scenario}/intervention_fortification/effective_coverage/{location}.csv"
    pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
    effective_intervention_coverage.reset_index().to_csv(path, index=False)

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.596000
                            2                rice            0.603684
                            3                rice            0.604358
                            4                rice            0.603568
                            5                rice            0.546544
        5          15       1                rice            0.651481
                            2                rice            0.654275
                            3                rice            0.652173
                            4                rice            0.634124
                            5                rice            0.560374
        15         30       1                rice            0.595837
                            2                rice            0.610998
                            3                rice            0.622165
                            4   

In [14]:
# Not all coverage is effective -- this is as a proportion of coverage!
effectiveness = pd.read_csv(
    f"{results_dir}/baseline_fortification/effectiveness/{location}.csv"
)
effectiveness = effectiveness.set_index(
    [c for c in effectiveness.columns if c != "value"]
).value

In [15]:
effective_baseline_coverage = any_consumption * current_coverage * effectiveness
effective_baseline_coverage

wealth_quintile  vehicle_name  sex     age_start  age_end
1                rice          Female  0          5          0.271771
                                       5          15         0.354580
                                       15         30         0.318263
                                       30         50         0.339757
                                       50         125        0.351941
                               Male    0          5          0.270326
                                       5          15         0.359366
                                       15         30         0.333086
                                       30         50         0.331810
                                       50         125        0.349362
2                rice          Female  0          5          0.265576
                                       5          15         0.350522
                                       15         30         0.312020
                                

In [16]:
path = f"{results_dir}/baseline_fortification/effective_coverage/{location}.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
effective_baseline_coverage.reset_index().to_csv(path, index=False)